# Hard-negative extension: comparison
Same script, same recipe, same seeds; the **only** thing that changes between arms is `--hard-neg-prob`
(probability a training negative is a narration from the *same video* as the audio clip).
Model selection is done on **val**; test is touched once at the very end.

In [ ]:
import importlib.util, shutil, time
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import DataLoader

# Import the script as a module (it only trains when main() is called)
spec = importlib.util.spec_from_file_location("atm", "audio-text_transformer_hn.py")
atm = importlib.util.module_from_spec(spec)
spec.loader.exec_module(atm)

DATA = Path("/data/COMS30017")
LR, EPOCHS = "1e-4", 10
SEEDS = [42, 43, 44]


In [ ]:
def train_run(tag, seed, hard_p, pos=False, scope="video"):
    """Train one model; saves best-val weights to <tag>_s<seed>.pth. Returns (best_val_acc, best_epoch)."""
    argv = ["--learning-rate", LR, "--epochs", str(EPOCHS), "--seed", str(seed),
            "--hard-neg-prob", str(hard_p), "--hard-scope", scope,
            "--model-name", f"{tag}_s{seed}", "--weights-out", f"{tag}_s{seed}.pth",
            "--print-frequency", "100000"]          # keep output quiet
    if pos:
        argv.append("--pos-embed")
    return atm.main(atm.parser.parse_args(argv))


@torch.no_grad()
def evaluate(weights_path, split="val", use_pos=False):
    audio, text, labels = atm.load_features(DATA, split)
    ds = atm.AudioTextDataset(audio, text, labels, training=False)
    loader = DataLoader(ds, batch_size=128, shuffle=False)
    a0, t0, _ = ds[0]
    m = atm.AudioTextTransformer(audio_dim=a0.shape[-1], text_dim=t0.shape[-1],
                                 d_model=atm.MODEL_DIM, num_heads=4,
                                 mlp_dim=atm.TRANSFORMER_MLP_DIM, dropout=atm.DROPOUT,
                                 use_pos=use_pos)
    m.load_state_dict(torch.load(weights_path, map_location=atm.DEVICE, weights_only=True))
    m.to(atm.DEVICE).eval()
    correct = total = 0
    for a, t, y in loader:
        preds = m(a.to(atm.DEVICE), t.to(atm.DEVICE)).argmax(-1).cpu()
        correct += (preds == y).sum().item()
        total += len(y)
    return correct / total


## Experiment: baseline (p=0) vs hard negatives (p=0.5, p=1.0)
9 runs total. Reports **validation** accuracy of each run's best-val checkpoint.

In [ ]:
CONFIGS = {            # tag -> hard-negative probability
    "base":  0.0,      # original uniform negatives
    "hn050": 0.5,      # half of negatives from the same video
    "hn100": 1.0,      # all negatives from the same video
}

val_results = {tag: [] for tag in CONFIGS}
for seed in SEEDS:
    for tag, p in CONFIGS.items():
        t0 = time.time()
        train_run(tag, seed, p)
        acc = evaluate(f"{tag}_s{seed}.pth", "val")
        val_results[tag].append(acc)
        print(f">>> {tag} seed {seed}: VAL acc {acc*100:.2f}%  ({time.time()-t0:.0f}s)\n")

print("\n=== Validation accuracy (mean ± std over seeds) ===")
for tag, accs in val_results.items():
    print(f"{tag:6s} p={CONFIGS[tag]:<4}  {np.mean(accs)*100:.2f}% ± {np.std(accs)*100:.2f}   runs={[round(a*100,2) for a in accs]}")


## Final test evaluation (run once, after choosing the config using val above)
Set `BEST_TAG` to the winning config. The baseline is evaluated alongside for the report table.

In [ ]:
BEST_TAG = "hn050"   # <- change to whichever tag won on VAL

print("=== TEST accuracy ===")
for tag in ("base", BEST_TAG):
    accs = [evaluate(f"{tag}_s{s}.pth", "test") for s in SEEDS]
    print(f"{tag:6s} {np.mean(accs)*100:.2f}% ± {np.std(accs)*100:.2f}   runs={[round(a*100,2) for a in accs]}")
